# Goal 2：单项候选、反例与阶段案例

C-S、C-D、C-P 分别只改变一个参数组，保留开发选择和 G2_EVAL 确认的区别。没有独立真值的自然数据只支持条件化覆盖、几何与约束结论。构造反例具有明确解析答案，其误删/漏检计数不能外推到真实数据。

In [ ]:
from pathlib import Path
import sys, json, tempfile, shutil
from html import escape
from IPython.display import display, Image, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'task1').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from task1.workflow.io import read_json, digest, object_hash
from task1.workflow.g2_provider import ExperimentProvider
model_sentinel = {'calls': 0}
def reject_model_call(*args, **kwargs):
    model_sentinel['calls'] += 1
    raise RuntimeError('RECOMPUTE_FORBIDS_NEW_MODEL_CALLS')
ExperimentProvider.call = reject_model_call
EV = ROOT/'task1/evidence/goal2'
current = read_json(EV/'current_runs.json')
contract = read_json(ROOT/'task1/config/goal2/contract.json')
split = read_json(EV/'data/split_manifest.json')
MODE = 'RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'RECOMPUTE' and ENABLE_LIVE is False, '新的 LIVE 调用须使用独立命令并显式启用，默认 Notebook 不发起模型请求。'
WORK = Path(tempfile.mkdtemp(prefix='sc-g2-notebook-'))
def show(rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = ''.join('<th>'+escape(str(key))+'</th>' for key in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(row.get(key, '')))+'</td>' for key in columns)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))
def figure(name):
    directory = Path(recomputed.get('figure_directory', WORK/'figures'))
    target = directory/(name+'.png')
    assert target.is_file(), f'复算图缺失: {target}'
    display(Image(filename=str(target), width=1050))
print('运行方式:', MODE, '| 新模型调用:', 0)
print('原始来源 datum:', contract['source_crs'], '| 单位: 工作米 / 原始相对秒')
print('当前固定分区:', {name:len(ids) for name,ids in split['splits'].items()})


## 从 raw 复算候选确认与反例

固定候选后才打开 G2_EVAL。本段重新执行对应真实数据处理与构造反例，检查保存结果一致；它是复现，不产生新的选择机会或独立实验数量。

In [ ]:
from task1.scripts.goal2 import recompute
recomputed = recompute(topic='candidates', output_directory=WORK, rebuild_figures=True)
assert recomputed['status'] == 'VERIFIED', recomputed
assert recomputed['new_model_calls'] == 0, recomputed
assert model_sentinel['calls'] == 0, model_sentinel
print('从 raw 和冻结提议复算:', recomputed['status'])
print('新增模型调用:', recomputed['new_model_calls'])
print('检查项数量:', len(recomputed['checks']))
show(recomputed['checks'][:20])
print('全部检查结果保存在独立复算输出，不把重复计算计为新独立实验。')


In [ ]:
lock = read_json(EV/'candidate_lock.json')
assert lock['source_partition']=='DEVELOPMENT' and lock['g2_eval_observed_for_selection'] is False
show([{'candidate':name,**row} for name,row in lock['selected_single_candidates'].items()])
print('额外结构候选:',lock['structural_candidates'])
print('仅以下冻结顺序进入阶段留出:',lock['feasible_orders'])

## 解析构造反例

正常转弯、合法 zigzag 和注入毛刺说明方向几何不是噪声真值。零位移和零时间差保留不可计算原因。有限线段反例拒绝无限直线捷径；P 先删触发点可能掩盖原始断点。整条无输出无法凭缺少异常获得好分，各自 clean 的零 DP 误差不能证明跨方法整体更好。这里检验风险，不伪造实际模型错误或用户质疑。

In [ ]:
counter_directory = ROOT/current['counterexamples']
synthetic = read_json(counter_directory/'synthetic_cases.json')
assert synthetic['classification']=='SYNTHETIC_COUNTEREXAMPLE'
show([{'case':case['case_id'],'family':case['family'],'checks':len(case['known_answer_checks']),'passed':all(check['passed'] for check in case['known_answer_checks']),'interpretation':case['interpretation']} for case in synthetic['cases']])
figure('synthetic_metric_counterexamples')

## 真实案例：代表样本、退化与近阈值

出图前按原始跨度层中位代表、最大覆盖损失、最大共同误差、最小正 P 阈值间隔规则登记案例。图只在工作平面显示点和真实输出片段，不叠加未核实底图，不跨记录/分段连线。缺输出同样保留。

In [ ]:
figure('real_trajectory_cases')
pilot = read_json(counter_directory/'exposed_pilot_cases.json')
show([{'record_id':row['record_id'],'raw_points':row['raw_points'],'final_points':row['final_points'],'segments':[{'n':part['n_points'],'length_working_m':part['within_segment_length_working_m'],'reasons':part['filter_reasons']} for part in row['segments']]} for row in pilot['all_filtered_records_0_1_2']])
near=pilot['near_threshold_case']
print('已暴露pilot近阈值:',{key:near[key] for key in ('record_id','original_index','reference_bracketing_indices','actual_error_working_m')})
show([{'dp':threshold,**row} for threshold,row in near['before_after'].items()])

## 实际生产工作流与被评估系统

主线程负责父 Goal；A 安排冻结范围内实验，B 执行或修复，独立 C 从可信原始输入核验。确定性 GoalJournal 检查版本、依赖和回执。普通问题经过修复、复验、失效产物重建后恢复父任务；这些角色调用与四模式内部调用分开计数。

In [ ]:
figure('goal2_actual_architecture')
state=read_json(EV/'goal_state.json')
show([{'issue':key,'parent_task':row['parent_task'],'status':row['status'],'fact':row.get('fact'),'repairer':row.get('repairer'),'verifier':row.get('verifier')} for key,row in state.get('issues',{}).items()])
print('Notebook执行时journal状态:',state['status'])

## 下一阶段边界

本轮只保留支持、否定、权衡或证据不足的单项结果供 Goal 3 判断。组合、冻结最终方法、最终全量处理、两份正式报告定稿、Evidence Lock 和提交包均不在本工作 Notebook 执行。最终网页 GPT 二重审核不能由内部自检替代。

In [ ]:
print('原始数据不变:',digest(ROOT/contract['raw_path'])==contract['raw_sha256'])
print('本次新模型调用:',recomputed['new_model_calls'])
print('GPT_SECOND_REVIEW=PENDING; Submission=NOT_READY')
shutil.rmtree(WORK)